# Spontaneous Hemodynamics — PSD, spectral-power maps, and ROI correlations

**A standalone analysis independent of Step7 and Step6c.** The analysis uses saved Step6b geometry and PASS criteria, Step4 processed signals, and finalized REST windows. ROI and whole-brain PSD are arithmetic means of pixelwise spectra. Correlations use only homotopically pairable pixels from saved Step6b masks. Their filter and edge trimming for each 60-second segment are specified in the **main project YAML**, under `step_spontaneous_hemodynamics.correlations` (`filter_type`, `band_hz`, `lowpass_high_hz`, `trim_left_sec`, and `trim_right_sec`). First the complete segment is filtered, then its edges may be trimmed, and finally ROI Pearson correlation and Fisher z are calculated. PSD and power maps always use the full unfiltered temporal segments (after the shared preprocessing and optional detrending). The REST branch is also defined in YAML. The figure controls below **only change visualization**, not the calculated results.

- **PSD:** overview and a configurable plot (Awake on the left, Iso on the right), selecting an ROI or the whole brain, hemisphere, absolute or relative PSD, axis ranges, individual mouse colors, and optional mouse highlighting.
- **Power maps:** selectable frequency bands, colormap, shared Awake/Iso display limits, and a separate scale for the Iso − Awake contrast.
- **Correlations:** common colormap and display limits for all six matrices, based on mean Fisher z values converted back to Pearson r.

**Important filter distinction:** `bandpass` applies both limits in `band_hz` (e.g. 0.08–0.40 Hz); `lowpass` uses **only** `lowpass_high_hz` (e.g. 0.40 Hz), with no high-pass component. The accompanying project YAML currently selects `lowpass`. Actual settings should always be checked in the YAML and the saved `analysis_methods.csv`.

To view existing CSV/NPY results without running the analysis again, set `RUN_BATCH = False`.


## 1. Paths and execution mode

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Source-code location, following the other Step4/Step6/Step6b notebooks.
PROJECT_CODE_ROOT = Path(r"...\WIFIOPIA_PT2021")
SRC_DIR = PROJECT_CODE_ROOT / "src"

# If the kernel starts in src or a neighboring directory, find src automatically.
if not SRC_DIR.is_dir():
    for candidate in (Path.cwd(), Path.cwd() / "src", Path.cwd().parent / "src"):
        if (candidate / "step_spontaneous_hemodynamics.py").is_file():
            SRC_DIR = candidate.resolve()
            break
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {SRC_DIR}. Проверьте PROJECT_CODE_ROOT.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import step_spontaneous_hemodynamics as sh

PROJECT_CONFIG = Path(r"config.yaml")
# All spontaneous-hemodynamics settings are loaded from PROJECT_CONFIG.
RUN_BATCH = True       # False: read existing tables and maps without recomputing.
OVERWRITE = True
ONLY_MICE = None       # For example, ["m14"] or None.
ONLY_STEMS = None

print("Источник модуля:", Path(sh.__file__).resolve())
print("SRC:", SRC_DIR)
print("Project config:", PROJECT_CONFIG)
print("Step settings: PROJECT_CONFIG / step_spontaneous_hemodynamics")


## 2. Preflight: recording selection and Step6b PASS eligibility

In [ ]:
pf = sh.preflight(PROJECT_CONFIG, mice=ONLY_MICE, stems=ONLY_STEMS)
display(pf['eligibility'].head(30))
print("Выбранная REST-ветка:", pf['config']['step_spontaneous_hemodynamics']['rest_variant'])
print("Awake accepted REST windows:", len(pf['rest60']))
print("Awake/Iso stems:")
display(pf['index'].groupby('spontaneous_condition').agg(n_mice=('mouse_id','nunique'), n_stems=('stem','nunique')).reset_index())

# Correlation-only analysis settings (read directly from the shared project YAML)
corr_cfg = dict(sh.DEFAULT['correlations'])
corr_cfg.update(pf['config']['step_spontaneous_hemodynamics'].get('correlations', {}))
filter_type = str(corr_cfg['filter_type']).lower().strip()
filter_bounds = (corr_cfg['band_hz'] if filter_type == 'bandpass'
                 else [None, corr_cfg['lowpass_high_hz']])
print('Фильтр ROI-корреляций:', filter_type, 'Hz:', filter_bounds)
print('Обрезка ROI-корреляций (слева, справа), сек:',
      corr_cfg['trim_left_sec'], corr_cfg['trim_right_sec'])
print('Принцип: фильтрация ВСЕГО 60-с окна -> обрезка -> ROI-корреляции.')
print('PSD и карты мощности: полное 60-с окно, без этой фильтрации и обрезки.')
print('Источник окон: with_rest = Awake из rest_segments_final.csv, Iso по полным 60-с блокам Zarr;')
print('              without_rest = Awake/Iso по полным 60-с блокам Zarr.')


## 3. Run the analysis (if enabled)

In [ ]:
if RUN_BATCH:
    result = sh.execute(PROJECT_CONFIG, mice=ONLY_MICE, stems=ONLY_STEMS, overwrite=OVERWRITE)
    print("Построенные файлы:", result['figures'])
else:
    print("RUN_BATCH=False. Используются сохранённые результаты; запуск Step7/Step6c не требуется.")
paths = pf['paths']
table_dir = paths['tables']
figure_dir = paths['figures']
print('Папка результатов:', paths['root'])
# Old spectra_mouse.csv can be incompatible: it used PSD of the mean time trace.
# Do not display stale scientific figures as if they had the new definition.
contract_file = table_dir / 'analysis_methods.csv'
if (table_dir / 'spectra_mouse.csv').exists():
    if not contract_file.is_file():
        raise RuntimeError('В результатах нет analysis_methods.csv: вероятно, расчёт старой версии. '
                           'Пересчитайте с RUN_BATCH=True и OVERWRITE=True.')
    methods = pd.read_csv(contract_file)
    if (methods.empty or methods.iloc[0].get('psd_roi') != 'mean_per_pixel_hann_periodogram'
            or methods.iloc[0].get('correlation_roi_mask') != 'step6b_homotopic_pairable_bilateral_canonical_per_stem'):
        raise RuntimeError('Сохранённые PSD/корреляции не соответствуют новому методу. '
                           'Пересчитайте с RUN_BATCH=True и OVERWRITE=True.')
    # Do not display previously computed correlations with changed filter or edge trim.
    stored = methods.iloc[0]
    current_filter = str(corr_cfg['filter_type'])
    current_high = (float(corr_cfg['band_hz'][1]) if current_filter == 'bandpass'
                    else float(corr_cfg['lowpass_high_hz']))
    current_low = float(corr_cfg['band_hz'][0]) if current_filter == 'bandpass' else np.nan
    correct = (str(stored.get('correlation_filter_type', '')) == current_filter
        and np.isclose(pd.to_numeric(stored.get('correlation_filter_high_hz'), errors='coerce'), current_high)
        and np.isclose(pd.to_numeric(stored.get('correlation_trim_left_sec'), errors='coerce'), float(corr_cfg['trim_left_sec']))
        and np.isclose(pd.to_numeric(stored.get('correlation_trim_right_sec'), errors='coerce'), float(corr_cfg['trim_right_sec'])))
    if current_filter == 'bandpass':
        correct = correct and np.isclose(
            pd.to_numeric(stored.get('correlation_filter_low_hz'), errors='coerce'), current_low)
    if not correct:
        raise RuntimeError('Параметры фильтра/обрезки в сохранённых результатах НЕ совпадают с общим YAML. '
                           'Для новой корреляции пересчитайте RUN_BATCH=True, OVERWRITE=True.')
    display(methods)

## 4. Recording, window, and error summary tables

In [ ]:
def read_table(name):
    """Read an existing result CSV from table_dir or return an empty DataFrame."""
    path = table_dir / (name + '.csv')
    if not path.is_file():
        print('Нет таблицы:', path)
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

summary = read_table('run_summary')
qc = read_table('roi_stem_segment_qc')
display(summary)
if not qc.empty:
    display(qc.groupby(['roi','side','status']).agg(n_segments=('segment','count'),n_stems=('stem','nunique')).reset_index())
errors = read_table('errors')
if not errors.empty:
    display(errors)

## 5. Pixel-mean PSD — individual mouse curves and group means

In [ ]:
mouse_psd = read_table('spectra_mouse')
if not mouse_psd.empty:
    for kind in ['relative_psd', 'absolute_psd']:
        dst = figure_dir / f'PSD_{"relative" if kind=="relative_psd" else "absolute"}_individual_mice.png'
        sh.plot_psd(mouse_psd, dst, kind=kind)
        print(dst.name)
        display(Image(filename=str(dst)))
else:
    print('PSD пока не рассчитана. Сначала запустите RUN_BATCH=True.')

band_mouse = read_table('relative_band_power_mouse_values_60s')
if not band_mouse.empty:
    display(band_mouse[['state','mouse_id','roi','hemisphere',
                        'relative_power_0p05_0p10','relative_power_0p11_0p20', 'relative_power_0p21_0p40']].head(30))

### 5a. Select one ROI or the whole brain: Awake left, Iso right

Edit the settings at the beginning of the next cell and rerun it. **Each mouse has a distinct color**; the thick black line is the **mean across mice**. Set `PSD_HIGHLIGHT_MOUSE` to additionally emphasize one mouse with a thick colored line. Frequency grids are not interpolated: the plot reads existing `spectra_mouse.csv` without recomputing PSD.

For `whole_brain`, only `bilateral` is available. This is the original whole-brain reference spectrum, not separate left/right hemispheric masks.


In [ ]:
# ===== SELECTED PSD FIGURE SETTINGS =====
PSD_KIND = "relative_psd"           # "relative_psd" or "absolute_psd"
PSD_ROI = "whole_brain"                     # "whole_brain", "MOp", "MOs", "SS", "RSP"
PSD_HEMISPHERE = "bilateral"        # "left", "right", "bilateral" (whole_brain: bilateral)
PSD_XLIM = (0.05, 0.5)              # Frequency in Hz, e.g. (0.02, 0.25).
PSD_YLIM = (0, 15)             # Shared Y-axis limits for Awake and Iso: (0, 30) or (None, None).
PSD_Y_SCALE = "linear"              # "linear" or "log"
PSD_HIGHLIGHT_MOUSE = None          # For example, "m14" highlights an individual curve; None disables highlighting.
PSD_SHOW_LEGEND = True
PSD_MEAN_LINEWIDTH = 3.5           # Thick black group-mean curve.
PSD_MOUSE_LINEWIDTH = 1.2
PSD_FIGSIZE = (14, 4.9)

# ===== PLOT FROM THE SAVED TABLE =====
import matplotlib.pyplot as plt

if PSD_KIND not in {"relative_psd", "absolute_psd"}:
    raise ValueError("PSD_KIND должен быть relative_psd или absolute_psd")
if PSD_ROI not in {"whole_brain", "MOp", "MOs", "SS", "RSP"}:
    raise ValueError(f"Неизвестная ROI: {PSD_ROI}")
if PSD_HEMISPHERE not in {"left", "right", "bilateral"}:
    raise ValueError(f"Неизвестное полушарие: {PSD_HEMISPHERE}")
if PSD_ROI == "whole_brain" and PSD_HEMISPHERE != "bilateral":
    raise ValueError("whole_brain хранится только с hemisphere='bilateral'")
if PSD_Y_SCALE not in {"linear", "log"}:
    raise ValueError("PSD_Y_SCALE должен быть 'linear' или 'log'")
if not PSD_XLIM[0] < PSD_XLIM[1]:
    raise ValueError("PSD_XLIM: min должен быть меньше max")
if all(v is not None for v in PSD_YLIM) and not PSD_YLIM[0] < PSD_YLIM[1]:
    raise ValueError("PSD_YLIM: min должен быть меньше max")

psd_source = mouse_psd if "mouse_psd" in globals() else read_table("spectra_mouse")
if psd_source.empty:
    print("Нет spectra_mouse.csv: сначала выполните расчёт или выберите папку результатов")
else:
    d = psd_source[
        (psd_source["psd_kind"].astype(str) == PSD_KIND)
        & (psd_source["roi"].astype(str) == PSD_ROI)
        & (psd_source["hemisphere"].astype(str) == PSD_HEMISPHERE)
        & (psd_source["state"].astype(str).isin(["Awake", "Iso"]))
    ].copy()
    if d.empty:
        print(f"Нет PSD для {PSD_KIND}, {PSD_ROI}, {PSD_HEMISPHERE}")
    else:
        d["frequency_hz"] = pd.to_numeric(d["frequency_hz"], errors="coerce")
        d["value"] = pd.to_numeric(d["value"], errors="coerce")
        d = d.dropna(subset=["frequency_hz", "value"])
        mouse_names = sorted(d["mouse_id"].astype(str).unique())
        palette = plt.get_cmap("tab20", max(1, len(mouse_names))) if len(mouse_names) <= 20 else plt.get_cmap("gist_ncar", len(mouse_names))
        mouse_colors = {mouse: palette(i) for i, mouse in enumerate(mouse_names)}
        fig, axes = plt.subplots(1, 2, figsize=PSD_FIGSIZE, sharex=True, sharey=True)
        for ax, state in zip(axes, ("Awake", "Iso")):
            gstate = d[d["state"].astype(str).eq(state)]
            for mouse, gm in gstate.groupby("mouse_id", sort=True):
                gm = gm.sort_values("frequency_hz")
                mouse = str(mouse)
                is_highlight = PSD_HIGHLIGHT_MOUSE is not None and mouse == str(PSD_HIGHLIGHT_MOUSE)
                ax.plot(gm["frequency_hz"].to_numpy(float), gm["value"].to_numpy(float),
                        color=mouse_colors[mouse],
                        lw=3.0 if is_highlight else PSD_MOUSE_LINEWIDTH,
                        alpha=1.0 if is_highlight else 0.83,
                        zorder=3 if is_highlight else 2, label=mouse)
            if not gstate.empty:
                # Each mouse contributes equally to the mean regardless of its segment count.
                piv = gstate.pivot_table(index="mouse_id", columns="frequency_hz", values="value", aggfunc="mean")
                x = piv.columns.to_numpy(float)
                ymean = np.nanmean(piv.to_numpy(float), axis=0)
                ax.plot(x, ymean, color="black", lw=PSD_MEAN_LINEWIDTH,
                        zorder=5, label=f"Среднее (n={len(piv)})")
            else:
                ax.text(0.5, 0.5, "Нет данных", transform=ax.transAxes,
                        ha="center", va="center")
            ax.set_title(f"{state} · {PSD_ROI} · {PSD_HEMISPHERE} · n={gstate.mouse_id.nunique()}")
            ax.set_xlim(*PSD_XLIM)
            ax.set_yscale(PSD_Y_SCALE)
            if any(v is not None for v in PSD_YLIM):
                ax.set_ylim(*PSD_YLIM)
            ax.set_xlabel("Частота, Гц")
            ax.grid(alpha=0.16)
            if PSD_SHOW_LEGEND and not gstate.empty:
                ax.legend(title="Мыши / среднее", fontsize=8, title_fontsize=8,
                          loc="best", ncol=2 if gstate.mouse_id.nunique() > 7 else 1)
        axes[0].set_ylabel("Относительная PSD (1/Гц)" if PSD_KIND == "relative_psd"
                           else "Абсолютная PSD (dcHbT²/Гц)")
        fig.suptitle("Индивидуальные спектры и групповое среднее (мышь = 1 наблюдение)")
        fig.tight_layout()
        figure_dir.mkdir(parents=True, exist_ok=True)
        suffix = f"{PSD_KIND}_{PSD_ROI}_{PSD_HEMISPHERE}"
        selected_psd_path = figure_dir / f"PSD_SELECTED_{suffix}.png"
        fig.savefig(selected_psd_path, dpi=190, bbox_inches="tight")
        plt.close(fig)
        print("Сохранено:", selected_psd_path)
        display(Image(filename=str(selected_psd_path)))


## 6. Previously computed normalized spectral-power maps

The original band definitions are preserved: **0.05–0.10 Hz** and **0.11–0.20 Hz** power relative to the integral of PSD over **0.05–0.50 Hz**. Other bands are available when saved in the configured map set. Pixelwise maps are computed from the same Hann-periodogram definition, without temporal band-pass filtering.

The controls below adjust `MAP_CMAP`, `MAP_VMIN`, and `MAP_VMAX` (shared for Awake and Iso); the difference map has separate `MAP_DIFF_*` limits. These parameters affect **visualization only**: the saved NPY maps are not modified.


In [ ]:
# ===== RELATIVE SPECTRAL-POWER MAP SETTINGS =====
MAP_BANDS = ["0p05_0p10", "0p11_0p20", "0p21_0p40",  "0p08_0p40"]  # Keep one band or add "0p08_0p40".
MAP_CMAP = "turbo"                   # For example: "viridis", "magma", "plasma", "turbo".
MAP_VMIN = 0.2                      # Shared minimum for Awake and Iso.
MAP_VMAX = 0.4                      # Shared maximum for Awake and Iso.
MAP_SHOW_DIFFERENCE = True          # Show Iso − Awake in the third column.
MAP_DIFF_CMAP = "coolwarm"          # Colormap for the difference.
MAP_DIFF_VMIN = -0.2                # Shared difference-map minimum.
MAP_DIFF_VMAX = 0.2                 # Shared difference-map maximum.

# CSV/NPY values are NOT clipped: these settings affect visualization only.
import matplotlib.pyplot as plt
import matplotlib as mpl

if not MAP_VMIN < MAP_VMAX:
    raise ValueError("MAP_VMIN должен быть меньше MAP_VMAX")
if MAP_SHOW_DIFFERENCE and not MAP_DIFF_VMIN < MAP_DIFF_VMAX:
    raise ValueError("MAP_DIFF_VMIN должен быть меньше MAP_DIFF_VMAX")
for name in (MAP_CMAP, MAP_DIFF_CMAP):
    if name not in mpl.colormaps:
        raise ValueError(f"Неизвестный colormap: {name}")

group_maps = read_table("group_map_index")
if group_maps.empty:
    print("Групповые карты пока отсутствуют (group_map_index.csv)")
else:
    display(group_maps)
    states = ["Awake", "Iso"] + (["Iso_minus_Awake"] if MAP_SHOW_DIFFERENCE else [])
    nrows, ncols = len(MAP_BANDS), len(states)
    if not nrows:
        raise ValueError("MAP_BANDS не должен быть пуст")
    fig, axs = plt.subplots(nrows, ncols, figsize=(5.15*ncols, 4.2*nrows), squeeze=False)
    for i, band in enumerate(MAP_BANDS):
        for j, state in enumerate(states):
            ax = axs[i, j]
            hits = group_maps[(group_maps["band"].astype(str) == band) &
                              (group_maps["state"].astype(str) == state)]
            if len(hits) > 1:
                raise ValueError(f"Повторная карта для {band}/{state}: {len(hits)}")
            if hits.empty:
                ax.text(0.5, 0.5, "Нет сохранённой карты", ha="center", va="center",
                        transform=ax.transAxes)
                ax.set_title(f"{state} · {band}")
            else:
                data = np.load(str(hits.iloc[0]["path"]))
                difference = state == "Iso_minus_Awake"
                cmap = MAP_DIFF_CMAP if difference else MAP_CMAP
                vmin, vmax = (MAP_DIFF_VMIN, MAP_DIFF_VMAX) if difference else (MAP_VMIN, MAP_VMAX)
                im = ax.imshow(np.ma.masked_invalid(data), cmap=cmap, vmin=vmin, vmax=vmax,
                               interpolation="nearest")
                title = "Iso − Awake" if difference else f"{state} · n={int(hits.iloc[0]['n_mice'])}"
                ax.set_title(f"{title} · {band}")
                fig.colorbar(im, ax=ax, fraction=0.043, pad=0.025)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Относительная спектральная мощность: PSD полосы / PSD 0.05–0.50 Гц")
    fig.tight_layout()
    figure_dir.mkdir(parents=True, exist_ok=True)
    custom_map_path = figure_dir / "relative_band_maps_CUSTOM.png"
    fig.savefig(custom_map_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print("Сохранено:", custom_map_path)
    display(Image(filename=str(custom_map_path)))


## 7. Correlations between MOp, MOs, SS, and RSP

Zero-lag Pearson correlations are computed within the original 60-second segments using only Step6b homotopically pairable pixels. The complete detrended segment is **filtered according to the main YAML**: either `bandpass` at `correlations.band_hz` or `lowpass` at `correlations.lowpass_high_hz` (the accompanying configuration uses low-pass **≤ 0.40 Hz**, without a 0.08-Hz high-pass cutoff). Optional `trim_left_sec` and `trim_right_sec` margins are removed **after** filtering, for correlations only. Fisher z is then averaged from segments to recording stems and subsequently to mice. Three matrices are provided for each condition: **left**, **right**, and **bilateral_trace_mean**. The bilateral variant averages left and right *temporal traces before correlation*; it is **not** the mean of two correlations.

`CORR_CMAP`, `CORR_VMIN`, and `CORR_VMAX` use the same display scale for **all six matrices**. Colors encode Pearson r back-transformed from the group mean of mouse-level Fisher z.


In [ ]:
# ===== SHARED DISPLAY SCALE FOR SIX CORRELATION MATRICES =====
CORR_CMAP = "coolwarm"               # For example: "RdBu_r", "seismic", "PuOr", "coolwarm".
CORR_VMIN = -1.0                     # Shared minimum for all six matrices (r scale).
CORR_VMAX = 1.0                      # Shared maximum for all six matrices (r scale).
CORR_SHOW_VALUES = True              # Label each cell with its r value.

import itertools
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize, TwoSlopeNorm

if not CORR_VMIN < CORR_VMAX:
    raise ValueError("CORR_VMIN должен быть меньше CORR_VMAX")
if CORR_CMAP not in mpl.colormaps:
    raise ValueError(f"Неизвестный colormap: {CORR_CMAP}")

corr_mouse = read_table("roi_correlation_mice")
if corr_mouse.empty:
    print("ROI-корреляции пока отсутствуют (roi_correlation_mice.csv)")
else:
    display(corr_mouse[["state", "mouse_id", "variant", "roi_a", "roi_b", "r", "fisher_z", "n_segments"]].head(30))
    corr_rois = ["MOp", "MOs", "SS", "RSP"]
    corr_modes = ["left", "right", "bilateral_trace_mean"]
    corr_states = ["Awake", "Iso"]
    norm = (TwoSlopeNorm(vmin=CORR_VMIN, vcenter=0.0, vmax=CORR_VMAX)
            if CORR_VMIN < 0 < CORR_VMAX else Normalize(vmin=CORR_VMIN, vmax=CORR_VMAX))
    fig, axes = plt.subplots(2, 3, figsize=(14.6, 8.2), squeeze=False)
    for i, state in enumerate(corr_states):
        for j, mode in enumerate(corr_modes):
            ax = axes[i, j]
            tmp = corr_mouse[(corr_mouse["state"].astype(str) == state) &
                             (corr_mouse["variant"].astype(str) == mode)]
            matrix = np.full((4, 4), np.nan, dtype=float)
            for roi in corr_rois:
                rows = tmp[(tmp["roi_a"] == roi) | (tmp["roi_b"] == roi)]
                if pd.to_numeric(rows["fisher_z"], errors="coerce").notna().any():
                    matrix[corr_rois.index(roi), corr_rois.index(roi)] = 1.0
            for a, b in itertools.combinations(corr_rois, 2):
                rows = tmp[(tmp["roi_a"] == a) & (tmp["roi_b"] == b)]
                z = pd.to_numeric(rows["fisher_z"], errors="coerce").dropna()
                if not z.empty:
                    # For display only, convert group-mean Fisher z back to r with tanh.
                    value = float(np.tanh(z.mean()))
                    ia, ib = corr_rois.index(a), corr_rois.index(b)
                    matrix[ia, ib] = matrix[ib, ia] = value
            im = ax.imshow(np.ma.masked_invalid(matrix), cmap=CORR_CMAP, norm=norm,
                           interpolation="nearest")
            ax.set_xticks(range(4), corr_rois)
            ax.set_yticks(range(4), corr_rois)
            ax.set_title(f"{state} · {mode} · n={tmp.mouse_id.nunique()}")
            if CORR_SHOW_VALUES:
                for rr in range(4):
                    for cc in range(4):
                        val = matrix[rr, cc]
                        if np.isfinite(val):
                            rgb = im.cmap(im.norm(val))[:3]
                            brightness = 0.2126*rgb[0] + 0.7152*rgb[1] + 0.0722*rgb[2]
                            ax.text(cc, rr, f"{val:.2f}", fontsize=8, ha="center", va="center",
                                    color="black" if brightness > 0.57 else "white")
    fig.subplots_adjust(left=0.075, right=0.855, bottom=0.07, top=0.92, hspace=0.32, wspace=0.34)
    cax = fig.add_axes([0.885, 0.13, 0.018, 0.72])
    fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=CORR_CMAP), cax=cax,
                 label="Среднее Fisher z, обратно преобразованное в Pearson r")
    figure_dir.mkdir(parents=True, exist_ok=True)
    custom_corr_path = figure_dir / "ROI_correlations_Awake_Iso_CUSTOM.png"
    fig.savefig(custom_corr_path, dpi=185, bbox_inches="tight")
    plt.close(fig)
    print("Сохранено:", custom_corr_path)
    display(Image(filename=str(custom_corr_path)))
    print("Отрицательные RSP-корреляции (если есть):")
    rsp = corr_mouse[(corr_mouse["roi_a"] == "RSP") | (corr_mouse["roi_b"] == "RSP")]
    display(rsp[rsp["r"] < 0][["state", "mouse_id", "variant", "roi_a", "roi_b", "r"]])


## 8. Review the outputs

For each ROI and hemisphere, examine `roi_stem_segment_qc.csv` for **Step6b cohort inclusion, stem-level Step6b PASS**, and a sufficient count of functional pixels. `status=PASS` applies to a specific stem and 60-second segment; failure for one mouse must not be interpreted as exclusion of that ROI from the entire cohort.

If RSP passes on both sides for a given mouse and all relevant paired-pixel and signal requirements are met, it should be available in the three correlation variants. Final geometry validation requires execution with the experimental Step4 Zarr datasets.

**Recompute results after changing to pixelwise PSD and paired-pixel masks** (`RUN_BATCH=True`, `OVERWRITE=True`): older CSV files are not scientifically compatible with the newer definitions. Recomputing writes output files; `OVERWRITE=True` explicitly permits replacement.

**QC check:** `roi_stem_segment_qc.csv` distinguishes `analysis=psd` (ordinary eligible ROI pixels) from `analysis=correlation_homotopic` (ROI pixels also restricted by the saved paired-pixel mask). Correlation `n_pixels` is the number of paired-mask FC pixels actually contributing to the trace.
